# Predict


In [ ]:
import geopandas as gpd
import rasterio as rio
import numpy as np

import geoutils as gu
import subkart
import joblib


In [ ]:
gdf = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/Basisdata_15_More_og_Romsdal_25833_Dybdedata_Dybdeareal.geo.parquet")
gdf = subkart.features.depth_preprocess(gdf)

In [ ]:
marine_vanntyper = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/mdir/NyTypologi2022.geo.parquet").to_crs(gdf.crs)

## Predict for AOI

In [ ]:
gdf_bunn_moere = gpd.read_file("https://storage.googleapis.com/niva-geodata/MarintNaturKart/bunn_in_kommuner_sea.geojson")
gdf_clip = gdf_bunn_moere.to_crs(gdf.crs)

vec_clip = gu.Vector(gdf_clip)

In [ ]:

clipped_marine_vanntyper = gdf_clip.overlay(marine_vanntyper)

In [ ]:
clipped_gdf = gdf_clip.overlay(gdf)

In [ ]:
rasters, one_hot_types = subkart.features.build_basis_raster(clipped_gdf, clipped_marine_vanntyper)

In [ ]:
features, valid_attrs = subkart.features.stack(rasters, one_hot_types)

X = features[valid_attrs]

In [ ]:
classifier = joblib.load("../data_generated/classifier.joblib")

In [ ]:
Y_pred = classifier.predict(X)

In [ ]:
pred_map = np.full(rasters[0].data.shape[-2:], np.nan, dtype=np.float32)
pred_map[valid_attrs] = Y_pred.astype(np.float32)

In [ ]:

pred_raster = gu.Raster.from_array(pred_map, transform=rasters[0].transform, crs=rasters[0].crs)

In [ ]:
subkart.utils.plot_prediction_raster(pred_raster)

# Vectorize output

In [ ]:
pred_vec = pred_raster.polygonize()

In [ ]:
# Map raster_value (0/1) to BunnType using existing mapping_values
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
pred_vec["BunnType"] = pred_vec["raster_value"].map(reverse_map)

pred_vec["BunnType"]

In [ ]:
gdf_final = pred_vec.ds.dissolve(by="BunnType", as_index=False, method="coverage")

In [ ]:
gdf_final = gdf_final.to_crs("EPSG:25833")

In [ ]:
fname = subkart.utils.to_filename(f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "moere-og-romsdal", "latest", gdf_final.crs.to_epsg())
gdf_final = gdf_final.drop(columns=["depth_range", "class"], errors="ignore")
gdf_final.to_file(f"{fname}.geojson", driver="GeoJSON")

In [ ]:
gdf_final.to_file(f"{fname}.gpkg", layer="soft_hard_bottom", driver="GPKG")